In [ ]:
import pandas as pd
import numpy as np
pd.set_option('display.max_columns', None)


In [ ]:
data_pendapatan=pd.read_csv('../data/data_pendapatan_cleared_mei_november_2023.csv').drop(columns=['Unnamed: 0'])
data_piutang=pd.read_csv('../data/Piutang Mei - November 2023.csv')

In [ ]:
data_piutang['Description'].fillna('Unknown', inplace=True)
data_piutang['Description'].isna().sum()

In [ ]:
common_customers = data_pendapatan['Cust No.'].isin(data_piutang['Customer'])
count_common_customers = common_customers.sum()
count_common_customers

In [ ]:
a=data_piutang['Customer'].unique()
len(data_piutang['Customer'].unique())


In [ ]:
common_customers = pd.Series(a).isin(data_pendapatan['Cust No.'])
count_common_customers = common_customers.sum()
count_common_customers

In [ ]:
unique_descriptions = data_piutang.groupby('Customer')['Description'].unique()
unique_descriptions_df = unique_descriptions.reset_index()
unique_descriptions_df.isna().sum()

In [ ]:
unique_descriptions_df

In [ ]:
# unique_descriptions_df['Description'] = unique_descriptions_df['Description'].apply(lambda x: x.tolist() if isinstance(x, np.ndarray) else x)
unique_descriptions_df['Description'] = unique_descriptions_df['Description'].apply(lambda x: tuple(x) if isinstance(x, list) else x)
descriptions_count = unique_descriptions_df.groupby('Customer')['Description'].nunique()
customers_with_multiple_descriptions = descriptions_count[descriptions_count > 1]
customers_with_multiple_descriptions
## Nocustomer with more than 1 descriptions

In [ ]:
single_description_customers = unique_descriptions_df.groupby('Customer')['Description'].nunique() == 1
customers_with_single_description = single_description_customers[single_description_customers].index
customers_with_nan_description = unique_descriptions_df[
    unique_descriptions_df['Customer'].isin(customers_with_single_description) & 
    unique_descriptions_df['Description'].apply(lambda x: pd.isna(x) or (isinstance(x, tuple) and all(pd.isna(i) for i in x)))
]
customers_with_nan_description['Customer'].unique()

In [ ]:
unique_descriptions_df['Description'] = unique_descriptions_df['Description'].apply(
    lambda x: x[0] if isinstance(x, tuple) else x
)
unique_descriptions_df

In [ ]:
unique_descriptions_df['Customer']=unique_descriptions_df['Customer'].astype('int64')


In [ ]:
unique_descriptions_df.to_excel('./../data/customer_description_mapping.xlsx')

# Testing Merge

In [ ]:
merged_data = data_pendapatan.merge(unique_descriptions_df, how='left', right_on='Customer', left_on='Cust No.')
merged_data['Customer'].fillna(0,inplace=True)
merged_data['Customer']=merged_data['Customer'].astype('int64')

merged_data[['Cust No.','Customer']]